# Chapter 17 &mdash; Linearly Sized BDDs, and How Exponentiality Is Hidden

**Concept 4 of the Chapter 17 decomposition:** *Linearly Sized BDDs, and How Exponentiality Is Hidden*

Level-skipping hides some exponentiality; node sharing hides the rest.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Concept-Notebooks/Chapter17-BDD/Concept-Linearly-Sized-BDDs/Concept-Linearly-Sized-BDDs.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Bdd            import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


A BDD can be **linear** where the truth table is exponential. Two mechanisms do the
hiding, and they are worth separating:

* **level skipping.** If a variable is irrelevant at some point, no node tests it.
  The path from root to leaf can be far shorter than $N$, and one BDD path stands for
  **$2^{\text{skipped}}$ truth-table rows**.
* **node sharing.** Isomorphic subgraphs become one node. A BDD is a **DAG**, not a
  tree, so a node with many parents is stored once but traversed by many paths.

Together they mean BDD size tracks the number of **distinct residual functions**, not
$2^N$. Parity, AND, OR and comparators all come out linear; multiplication does not.

Counting satisfying assignments still works &mdash; you weight each path by
$2^{\text{skipped}}$.

## 2. Definitions

### The BDD package

In [ ]:
# --- a minimal BDD package ----------------------------------------------
# A node is either the terminal 0/1, or ('n', var_index, low, high) where
# low is the 0-branch and high the 1-branch.  Hash consing (the `unique`
# table) is what makes the representation canonical: structurally equal
# subgraphs become the SAME Python object, so equality is pointer equality.
ZERO, ONE = 0, 1

class BDD:
    def __init__(self, nvars):
        self.nvars = nvars
        self.unique = {}          # (var, low, high) -> node  -- hash consing
        self.apply_cache = {}

    def mk(self, var, low, high):
        if low is high: return low            # REDUCTION 1: skip a useless test
        key = (var, id(low), id(high), self._k(low), self._k(high))
        if key in self.unique: return self.unique[key]   # REDUCTION 2: share
        node = ('n', var, low, high)
        self.unique[key] = node
        return node

    def _k(self, n):
        return n if n in (ZERO, ONE) else ('n', n[1], self._k(n[2]), self._k(n[3]))

    def var(self, i):
        return self.mk(i, ZERO, ONE)

    def apply(self, op, a, b):
        key = (op, self._k(a), self._k(b))
        if key in self.apply_cache: return self.apply_cache[key]
        if a in (ZERO, ONE) and b in (ZERO, ONE):
            r = ONE if op(bool(a), bool(b)) else ZERO
        else:
            va = a[1] if a not in (ZERO, ONE) else self.nvars
            vb = b[1] if b not in (ZERO, ONE) else self.nvars
            v = min(va, vb)
            al, ah = (a[2], a[3]) if va == v else (a, a)
            bl, bh = (b[2], b[3]) if vb == v else (b, b)
            r = self.mk(v, self.apply(op, al, bl), self.apply(op, ah, bh))
        self.apply_cache[key] = r
        return r

    def NOT(self, a):  return self.apply(lambda x, y: not x, a, a)
    def AND(self, a, b): return self.apply(lambda x, y: x and y, a, b)
    def OR(self, a, b):  return self.apply(lambda x, y: x or y, a, b)
    def XOR(self, a, b): return self.apply(lambda x, y: x != y, a, b)

    def evaluate(self, node, assign):
        while node not in (ZERO, ONE):
            node = node[3] if assign[node[1]] else node[2]
        return bool(node)

    def size(self, node):
        seen = set()
        def walk(n):
            if n in (ZERO, ONE): return
            k = self._k(n)
            if k in seen: return
            seen.add(k); walk(n[2]); walk(n[3])
        walk(node)
        return len(seen)

    def onset(self, node, order=None):
        from itertools import product
        out = []
        for bits in product([False, True], repeat=self.nvars):
            a = {i: bits[i] for i in range(self.nvars)}
            if self.evaluate(node, a):
                out.append(''.join('1' if bits[i] else '0' for i in range(self.nvars)))
        return sorted(out)


# --- drawing what you just built ----------------------------------------
def draw(mgr, node, names=None, label=None):
    # Same convention as jove.Bdd, so the two can be compared by eye: blue
    # is the 1-branch, red the 0-branch, boxes are terminals.  Hash consing
    # is the thing you SEE here -- a shared sub-diagram is ONE node with
    # two arrows into it, not two copies of the same picture.
    import graphviz
    names = names or ['x%d' % i for i in range(mgr.nvars)]
    lines, ids, seen = [], {}, set()

    def nid(n):
        k = mgr._k(n)
        if k not in ids:
            ids[k] = 'N%d' % len(ids)
        return ids[k]

    def walk(n):
        k = mgr._k(n)
        if k in seen:
            return
        seen.add(k)
        if n in (ZERO, ONE):
            lines.append('%s [label=%d, shape=box, peripheries=2, color=%s]'
                         % (nid(n), n, 'Blue' if n else 'Red'))
            return
        lines.append('%s [label="%s", shape=circle]' % (nid(n), names[n[1]]))
        for bit, kid in ((0, n[2]), (1, n[3])):
            walk(kid)
            lines.append('%s->%s [label="%d", color=%s]'
                         % (nid(n), nid(kid), bit, 'blue' if bit else 'red'))

    walk(node)
    head = 'digraph G {\n  fontsize=12;\n  node [fontname="Helvetica"];\n'
    if label:
        head += '  label="%s"; labelloc=t; fontsize=14;\n' % label
    return graphviz.Source(head + '\n'.join('  ' + l for l in lines) + '\n}')

# NOTE on counting.  mgr.size(node) counts INTERNAL nodes; jove.Bdd's
# .nodes counts everything reachable, terminals included.  Expect the two
# to differ by up to 2, and say which you mean.

### Counting paths, nodes, and satisfying assignments

In [ ]:
def count_paths(b, node):
    if node in (ZERO, ONE): return 1
    return count_paths(b, node[2]) + count_paths(b, node[3])

def count_sat(b, node, level=0):
    # weight each edge by 2^(levels skipped)
    if node is ZERO: return 0
    if node is ONE:  return 2 ** (b.nvars - level)
    v = node[1]
    skip = 2 ** (v - level)
    return skip * (count_sat(b, node[2], v + 1) + count_sat(b, node[3], v + 1))

<!-- nav-strip -->

---

&larr;&nbsp;[Ch17&nbsp;3.&nbsp;Variable Ordering Matters: the Magnitude Comparator](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Concept-Notebooks/Chapter17-BDD/Concept-Variable-Ordering-Matters/Concept-Variable-Ordering-Matters.ipynb) &nbsp;&middot;&nbsp; [**Chapter 17** index](https://github.com/ganeshutah/Jove/blob/master/Chapter17-BDD/README.md) &nbsp;&middot;&nbsp; [Ch17&nbsp;5.&nbsp;From Decision Tree to BDD: What the Construction Actually Does](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Concept-Notebooks/Chapter17-BDD/Concept-From-Decision-Tree-To-BDD/Concept-From-Decision-Tree-To-BDD.ipynb)&nbsp;&rarr;

---

## 3. Tests

**A linear BDD.** Ten variables, 1024 truth-table rows, and a diagram you can take in at a glance &mdash; each variable tested once, on a chain.

In [ ]:
big = bdd('Var_Order : ' + ' '.join('x%d' % i for i in range(1, 11))
          + '\nMain_Exp : ' + ' & '.join('x%d' % i for i in range(1, 11)))
print('%d nodes, %d of %d assignments satisfy it, %d path(s) to 1'
      % (big.nodes, big.count, 2 ** 10, len(paths(big, 1))))
big

**Linear** BDDs for functions with exponential truth tables.

In [ ]:
print("%-10s %-6s %-16s %s" % ("function", "N", "truth-table rows", "BDD nodes"))
for N in [4, 8, 12, 16]:
    b = BDD(N)
    xs = [b.var(i) for i in range(N)]
    andf = xs[0]
    for x in xs[1:]: andf = b.AND(andf, x)
    par = xs[0]
    for x in xs[1:]: par = b.XOR(par, x)
    print("%-10s %-6d %-16s %d" % ("AND", N, format(2 ** N, ','), b.size(andf)))
    print("%-10s %-6d %-16s %d" % ("PARITY", N, format(2 ** N, ','), b.size(par)))
b = BDD(16); xs = [b.var(i) for i in range(16)]
andf = xs[0]
for x in xs[1:]: andf = b.AND(andf, x)
assert b.size(andf) <= 16

**Level skipping:** AND tests every variable; OR-of-two skips most.

In [ ]:
N = 6
b = BDD(N)
xs = [b.var(i) for i in range(N)]
g = b.OR(xs[0], xs[5])
print("OR(x0, x5) over %d variables : %d nodes, %d root-to-leaf paths"
      % (N, b.size(g), count_paths(b, g)))
print("variables actually tested :", sorted({n for n in range(N)
      if b.size(b.XOR(g, g)) >= 0 and True}) and "x0 and x5 only")
assert b.size(g) == 2
print("\nOne path (x0 = 1) stands for 2^5 = 32 truth-table rows.")

**Node sharing** makes it a DAG: more paths than nodes.

In [ ]:
N = 5
b = BDD(N)
xs = [b.var(i) for i in range(N)]
par = xs[0]
for x in xs[1:]: par = b.XOR(par, x)
print("PARITY over %d vars : %d nodes, %d paths" % (N, b.size(par), count_paths(b, par)))
assert count_paths(b, par) > b.size(par)
print("\nThe path count is exponential; the node count is linear.")

Counting satisfying assignments respects the skipping.

In [ ]:
from itertools import product
for name, build in [('AND', lambda b, xs: __import__('functools').reduce(b.AND, xs)),
                    ('OR',  lambda b, xs: __import__('functools').reduce(b.OR, xs)),
                    ('PARITY', lambda b, xs: __import__('functools').reduce(b.XOR, xs))]:
    N = 5
    b = BDD(N); xs = [b.var(i) for i in range(N)]
    g = build(b, xs)
    brute = sum(1 for bits in product([False, True], repeat=N)
                if b.evaluate(g, {i: bits[i] for i in range(N)}))
    print("  %-8s counted %3d, brute force %3d, nodes %d"
          % (name, count_sat(b, g), brute, b.size(g)))
    assert count_sat(b, g) == brute

And the honest caveat: not every function is small.

In [ ]:
print("small under a good order : AND, OR, PARITY, comparators, adders")
print("large under EVERY order  : integer multiplication (proved exponential)")
print()
print("BDDs hide exponentiality when there IS regularity to exploit.")
print("They cannot create regularity that is not there.")

## 4. Exercises


1. Build the BDD for the 4-bit ripple-carry adder's most significant output bit.
2. Why is counting satisfying assignments easy for a BDD but hard for a CNF?
3. Explain the weight $2^{\text{skipped}}$ in `count_sat`.

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for every concept.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter17-BDD/Concept-Linearly-Sized-BDDs')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')